# Tests des fonctions de calcul de la saturation

In [1]:
import json
from datetime import date

import pandas as pd
from saturation_image_quali import (
    get_sampled_state_poc,
    to_sampled_state_grp,
    to_state_grp_d,
    to_state_grp_h,
    to_state_poc_d,
)

SAMPLES: int = 288  # 5 min
SATURE_H: int = 45  # minimum duration (min) of saturation to have a saturated hour

ID_POC: str = "id_pdc_itinerance"
ID_STATION: str = "id_station_itinerance"
SATURATION_RATIO = 0.1
OVERLOAD_RATIO = 0.2
MIN_POWER = 75

day = date(2026,7, 5)
date_calcul = "2026-07-05"
date_file = date_calcul.replace("-", "")

data_quali = "../data/"

In [2]:
def read_statics(day: date, min_power: float) -> pd.DataFrame:
    """Read static data for POC and stations."""
    e5_str = pd.read_csv("../data_DMR_e2_e3/e5_05-07-2026.csv")["extras"][0]
    statics = pd.DataFrame(json.loads(e5_str))
    statics["unite"] = statics["id_pdc_itinerance"].str[:5]
    return statics[statics["puissance_nominale"] >= min_power]


In [3]:
sessions_s3 = pd.read_parquet(data_quali + "qualicharge-" + date_file + "/sessions/production.parquet", engine="pyarrow")
statuses_s3 = pd.read_parquet(data_quali + "qualicharge-" + date_file + "/statuses/production.parquet", engine="pyarrow")

In [4]:
sessions = pd.read_csv('../data_test/donnees_sessions_FRHPCPNF050462_05-07-2026.csv')[['start', 'end', 'id_pdc_itinerance']]
sessions['start'] = pd.to_datetime(sessions['start'])
sessions['end'] = pd.to_datetime(sessions['end'])
statuses = pd.DataFrame({ID_POC:[], "horodatage":[], "etat_pdc":[]})
statuses['horodatage'] = pd.to_datetime(statuses['horodatage'], utc=True)
# statuses = pd.DataFrame()
statics = pd.DataFrame({ID_POC:['FRHPCENF050462001', 'FRHPCENF050462002', 'FRHPCENF050462004' ], ID_STATION:['FRHPCPNF050462']*3})

samples_per_day = SAMPLES #72 #288

e2_str = pd.read_csv("../data_DMR_e2_e3/e2_e3_05-07-2026.csv")["extras"][0]
e2_pdc = pd.DataFrame(json.loads(e2_str))

e3_str = pd.read_csv("../data_DMR_e2_e3/e2_e3_05-07-2026.csv")["extras"][1]
e3_station = pd.DataFrame(json.loads(e3_str))

e5_statics = read_statics(day, MIN_POWER)

# FRHPCENF050462001
e2_pdc = e2_pdc[e2_pdc[ID_POC].str[:14] == 'FRHPCENF050462']
e3_station = e3_station[e3_station[ID_STATION] == 'FRHPCPNF050462']


In [5]:
e5_statics[e5_statics[ID_STATION] == 'FRHPCPNF050462']

,latitude,longitude,id_pdc_itinerance,puissance_nominale,id_station_itinerance,unite
6173,48.550222,-3.112511,FRHPCENF050462001,300.0,FRHPCPNF050462,FRHPC
10378,48.550222,-3.112511,FRHPCENF050462003,300.0,FRHPCPNF050462,FRHPC
19057,48.550222,-3.112511,FRHPCENF050462002,300.0,FRHPCPNF050462,FRHPC
19767,48.550222,-3.112511,FRHPCENF050462004,300.0,FRHPCPNF050462,FRHPC


## test local

In [6]:
samples_per_day = 288
sampled_state_poc = get_sampled_state_poc(day, samples_per_day, sessions, statuses)

In [7]:
sampled_state_poc[sampled_state_poc[ID_POC] == 'FRHPCENF050462001']

,id_pdc_itinerance,periode,state
0,FRHPCENF050462001,2026-07-05 00:00:00+00:00,libre
1,FRHPCENF050462001,2026-07-05 00:05:00+00:00,libre
2,FRHPCENF050462001,2026-07-05 00:10:00+00:00,libre
3,FRHPCENF050462001,2026-07-05 00:15:00+00:00,libre
4,FRHPCENF050462001,2026-07-05 00:20:00+00:00,libre
...,...,...,...
283,FRHPCENF050462001,2026-07-05 23:35:00+00:00,libre
284,FRHPCENF050462001,2026-07-05 23:40:00+00:00,libre
285,FRHPCENF050462001,2026-07-05 23:45:00+00:00,libre
286,FRHPCENF050462001,2026-07-05 23:50:00+00:00,libre


In [8]:
state_poc_d = to_state_poc_d(sampled_state_poc, samples_per_day)

In [9]:
state_poc_d, e2_pdc

(   id_pdc_itinerance  occupe  hors_service   libre
 0  FRHPCENF050462001    55.0           0.0  1385.0
 1  FRHPCENF050462002    50.0           0.0  1390.0
 2  FRHPCENF050462004   215.0           0.0  1225.0,
        libre  occupe  hors_service  id_pdc_itinerance
 1489   640.0   800.0           0.0  FRHPCENF050462001
 1490   640.0   800.0           0.0  FRHPCENF050462002
 1491  1220.0   220.0           0.0  FRHPCENF050462004)

In [10]:
sample_state_station = to_sampled_state_grp(sampled_state_poc, statics, ID_STATION, SATURATION_RATIO, OVERLOAD_RATIO)

state_station_h = to_state_grp_h(sample_state_station, ID_STATION, SAMPLES, SATURE_H)

state_station_d = to_state_grp_d(state_station_h, ID_STATION)

In [11]:
state_station_d


,id_station_itinerance,periode,nb_pdc,nb_h,hs,inactif,sature_cum,sature_max,surcharge,actif
0,FRHPCPNF050462,2026-07-05,3,24,0.0,1135.0,0.0,0.0,0.0,305.0


In [12]:
e3_station


,hs,nb_h,actif,nb_pdc,inactif,surcharge,sature_cum,sature_max,id_station_itinerance
354,0.0,24,650.0,4,585.0,0.0,205.0,55.0,FRHPCPNF050462


## test global

In [13]:
sessions_s3[sessions_s3[ID_POC] == 'FRHPCENF050462002']

,id_pdc_itinerance,energy,created_at,updated_at,id,start,end,point_de_charge_id,created_by_id,updated_by_id
164907,FRHPCENF050462002,40.508,2026-07-08 10:21:14.018578+00:00,2026-07-08 10:21:14.018593+00:00,ac90733a-0b8e-4349-a6b6-fcbb3d98ef7a,2026-07-05 09:26:25+00:00,2026-07-05 09:50:43+00:00,0220bf18-f66b-4d05-963f-2b5b28b66f12,179b8133-2e72-4551-89f0-1f34dc958dfb,<NA>
164910,FRHPCENF050462002,5.615,2026-07-08 10:21:13.828872+00:00,2026-07-08 10:21:13.828910+00:00,e91db5ff-93e7-4f26-b114-672400d503da,2026-07-05 11:26:47+00:00,2026-07-08 10:16:17+00:00,0220bf18-f66b-4d05-963f-2b5b28b66f12,179b8133-2e72-4551-89f0-1f34dc958dfb,<NA>
164914,FRHPCENF050462002,20.374,2026-07-08 10:20:13.518705+00:00,2026-07-08 10:20:13.518720+00:00,1b359e59-0871-4358-b672-3856dc82ede5,2026-07-05 07:27:39+00:00,2026-07-05 07:54:11+00:00,0220bf18-f66b-4d05-963f-2b5b28b66f12,179b8133-2e72-4551-89f0-1f34dc958dfb,<NA>


In [14]:
samples_per_day = 72
sampled_state_poc_g = get_sampled_state_poc(day, samples_per_day, sessions_s3, statuses_s3)

In [15]:
sampled_state_poc_g[sampled_state_poc_g[ID_POC] == 'FRHPCENF050462001']

,id_pdc_itinerance,periode,state
1110476,FRHPCENF050462001,2026-07-05 00:00:00+00:00,libre
1110477,FRHPCENF050462001,2026-07-05 00:20:00+00:00,libre
1110478,FRHPCENF050462001,2026-07-05 00:40:00+00:00,libre
1110479,FRHPCENF050462001,2026-07-05 01:00:00+00:00,libre
1110480,FRHPCENF050462001,2026-07-05 01:20:00+00:00,libre
...,...,...,...
1110543,FRHPCENF050462001,2026-07-05 22:20:00+00:00,libre
1110544,FRHPCENF050462001,2026-07-05 22:40:00+00:00,libre
1110545,FRHPCENF050462001,2026-07-05 23:00:00+00:00,libre
1110546,FRHPCENF050462001,2026-07-05 23:20:00+00:00,libre


In [16]:
state_poc_d_g = to_state_poc_d(sampled_state_poc_g, samples_per_day)

In [17]:
state_poc_d_g[state_poc_d_g[ID_POC].str[:14] == 'FRHPCENF050462']

,id_pdc_itinerance,occupe,hors_service,libre
15423,FRHPCENF050462001,80.0,0.0,1360.0
15424,FRHPCENF050462002,40.0,0.0,1400.0
15425,FRHPCENF050462004,100.0,0.0,1340.0


In [18]:
sample_state_station_g = to_sampled_state_grp(sampled_state_poc_g, e5_statics, ID_STATION, SATURATION_RATIO, OVERLOAD_RATIO)

state_station_h_g = to_state_grp_h(sample_state_station_g, ID_STATION, SAMPLES, SATURE_H)

state_station_d_g = to_state_grp_d(state_station_h_g, ID_STATION)

In [19]:
state_station_d_g[state_station_d_g[ID_STATION] == 'FRHPCPNF050462']

,id_station_itinerance,periode,nb_pdc,nb_h,hs,inactif,sature_cum,sature_max,surcharge,actif
354,FRHPCPNF050462,2026-07-05,1,24,0.0,310.0,0.0,0.0,0.0,50.0


In [20]:
state_station_d_g[state_station_d_g["sature_cum"] >= 50]

,id_station_itinerance,periode,nb_pdc,nb_h,hs,inactif,sature_cum,sature_max,surcharge,actif
217,FREVCP000170,2026-07-05,0,24,0.0,185.0,50.0,15.0,0.0,125.0
352,FRFASP11568651,2026-07-05,0,24,0.0,290.0,70.0,10.0,0.0,0.0
355,FRHPCPNF058911TOTEM,2026-07-05,0,24,0.0,250.0,110.0,15.0,0.0,0.0
357,FRHPCPNF059678,2026-07-05,0,24,0.0,300.0,60.0,10.0,0.0,0.0
361,FRHPCPNF080197TOTEMSA,2026-07-05,0,24,0.0,295.0,65.0,10.0,0.0,0.0
362,FRHPCPNF080266TOTEM,2026-07-05,0,24,0.0,225.0,50.0,10.0,0.0,85.0
363,FRIOYP13530793,2026-07-05,0,24,0.0,290.0,70.0,15.0,0.0,0.0
365,FRIOYP13531030,2026-07-05,0,24,0.0,220.0,140.0,15.0,0.0,0.0
366,FRIOYP13531057,2026-07-05,0,24,0.0,285.0,75.0,15.0,0.0,0.0
369,FRMBIPVGODS,2026-07-05,0,24,0.0,260.0,50.0,15.0,0.0,50.0


## Test échantillonage des sessions

In [21]:
echantillons = 24
timestamp = pd.Timestamp('2025-04-25T00:00:00+02:00')
start = [1, 1.2, 3, 5.5, 9, 13.1, 20]
end = [2.1, 2.7, 5, 7.5, 12.1, 15.1, 22.6]
test = pd.DataFrame( {'start': [timestamp + pd.Timedelta(hours=val) for val in start],
                      'end': [timestamp + pd.Timedelta(hours=val) for val in end],
                      'id_pdc_itinerance': ['p1', 'p2', 'p2', 'p1', 'p2', 'p1', 'p2']})
pdc = test['id_pdc_itinerance'].unique()
init = pd.DataFrame( {'start': [timestamp + pd.Timedelta(days=-1)] * len(pdc), 
                      'end': [timestamp + pd.Timedelta(hours=-1)] * len(pdc),
                      'id_pdc_itinerance': pdc}) 
# p1 : [1, 2.1], [5.5, 7.5], [13.1, 15.1]
# p2 : [1.2, 2.7], [3, 5], [9, 12.1], [20, 22.6]
sessions = to_sampled_sessions(test, init, timestamp, echantillons)

assert sessions.iloc[5]['occupation_pdc'] == 'f_libre'
assert sessions.iloc[6]['occupation_pdc'] == 'occupe'

end = [6.1, 2.7, 5, 7.5, 12.1, 15.1, 22.6]
test = pd.DataFrame( {'start': [timestamp + pd.Timedelta(hours=val) for val in start],
                      'end': [timestamp + pd.Timedelta(hours=val) for val in end],
                      'id_pdc_itinerance': ['p1', 'p2', 'p2', 'p1', 'p2', 'p1', 'p2']}) 
# p1 : [1, 6.1], [5.5, 7.5], [13.1, 15.1]
# p2 : [1.2, 2.7], [3, 5], [9, 12.1], [20, 22.6]
sessions = to_sampled_sessions(test, init, timestamp, echantillons)

assert sessions.iloc[5]['occupation_pdc'] == 'occupe'
assert sessions.iloc[6]['occupation_pdc'] == 'occupe'

sessions

NameError: name 'to_sampled_sessions' is not defined

## Test échantillonage des statuts

In [ ]:
echantillons = 24
timestamp = pd.Timestamp('2025-04-25T00:00:00+02:00')
valeurs = [1, 1.2, 3, 3.5, 5, 6.1, 12]

test = pd.DataFrame( {'horodatage': [timestamp + pd.Timedelta(hours=val) for val in valeurs],
                      'etat_pdc':['en_service', 'hors_service', 'en_service', 'en_service', 
                                  'hors_service', 'en_service', 'hors_service'],
                      'id_pdc_itinerance': ['p1', 'p2', 'p2', 'p1', 'p2', 'p1', 'p2']})
pdc = test['id_pdc_itinerance'].unique()
init = pd.DataFrame( {'horodatage': [timestamp + pd.Timedelta(days=-1)] * len(pdc), 
                      'etat_pdc': ['en_service'] * len(pdc), 
                      'id_pdc_itinerance': pdc}) 
statuses = to_sampled_statuses(test, init, timestamp, echantillons)
assert statuses.iloc[25]['etat_pdc'] == 'en_service'
assert statuses.iloc[26]['etat_pdc'] == 'hors_service'
statuses

## Test assemblage des sessions et des statuts

In [ ]:
sessions = pd.DataFrame({'id_pdc_itinerance': ['p1', 'p1', 'p1', 'p2', 'p2', 'p2', 'p3', 'p3', 'p3'], 
                       'periode': [0,1,2,0,1,2,0,1,2],
                       'occupation_pdc': ['occupe', 'f_libre', 'occupe', 'f_libre', 'occupe', 'f_libre','f_libre', 'occupe', 'f_libre']})
status = pd.DataFrame({'id_pdc_itinerance': ['p1', 'p1', 'p1', 'p3', 'p3', 'p3', 'p4', 'p4', 'p4'], 
                       'periode': [0,1,2,0,1,2, 0,1,2],
                       'etat_pdc': ['hors_service', 'hors_service', 'en_service', 'en_service', 'hors_service', 'hors_service', 'en_service', 'hors_service', 'en_service']})
merged = pd.merge(sessions, status, how='outer', on=['id_pdc_itinerance', 'periode']).fillna('aaa')
merged

In [ ]:
merged = to_sampled_state_pdc(sessions, status)
assert list(merged['state'][0:4]) == ['occupe', 'hors_service', 'occupe', 'libre']
merged

In [ ]:
print(merged['state'])
merged['state'].str.replace('en_service', 'libre')
merged['state'] = merged['state'].str.replace('en_service', 'libre')
merged

## Test état global échantillonné d'un groupement de pdc

In [ ]:
test = pd.DataFrame({'id_pdc_itinerance': ['p1', 'p1', 'p1', 'p2', 'p2', 'p2', 'p3', 'p3', 'p3'],
                     'periode' : [0, 1, 2, 0, 1, 2, 0, 1, 2],
                     'state' : ['occupe', 'hors_service', 'occupe', 'libre', 'occupe', 'libre', 'libre', 'occupe', 'hors_service']})
stations = pd.DataFrame({'id_pdc_itinerance': ['p1', 'p2', 'p3'],
                         'id_station_itinerance': ['s1', 's1', 's2']}) 
to_sampled_state_grp(test, stations, 'id_station_itinerance', 0.1, 0.2)

In [ ]:
test = pd.DataFrame({'id_pdc_itinerance': ['p1', 'p1', 'p1', 'p1', 'p1', 'p1',
                                           'p2', 'p2', 'p2', 'p2', 'p2', 'p2'],
                     'periode' : [0, 1, 2, 3, 4, 5,
                                  0, 1, 2, 3, 4, 5],
                     'state' : ['occupe', 'hors_service', 'occupe', 'occupe', 'hors_service', 'libre',
                                'libre', 'libre', 'occupe', 'hors_service', 'hors_service', 'libre']})
stations = pd.DataFrame({'id_pdc_itinerance': ['p1', 'p2'],
                         'id_station_itinerance': ['s1', 's1']}) 
to_sampled_state_grp(test, stations, 'id_station_itinerance', 0.1, 0.2)

In [ ]:
test = pd.DataFrame({'name':         ['hs', 'inactif', 'sature', 'surcharge', 'actif'],
                     'occupe':       [0, 0, 5, 3, 2],
                     'hors_service': [6, 2, 1, 2, 2],
                     'libre':        [0, 4, 0, 1, 2],
                     'nb_pdc':       [6, 6, 6, 6, 6]})
# 2e partie de la fonction : to_sampled_state_grp
test['hs'] = (test['libre'] + test['occupe'] == 0) & (test['hors_service'] > 0)
test['inactif'] = ~test['hs'] & (test['occupe'] == 0)
test['sature'] = ~test['hs'] & ~test['inactif'] & (test['libre']/test['nb_pdc'] < 0.1)
test['surcharge'] = ~test['hs'] & ~test['inactif'] & ~test['sature'] & (test['libre']/test['nb_pdc'] < 0.2)
test['actif'] = ~test['hs'] & ~test['inactif'] & ~test['sature'] & ~test['surcharge']
test['state'] = test['hs'] + test['inactif'] * 2 + test['actif'] * 3 + test['surcharge'] * 4 + test['sature'] * 5
test

In [ ]:
test = pd.DataFrame({'id_pdc_itinerance': ['pdc1'] * 10 + ['pdc2'] * 10,
                     'periode':      [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] * 2,
                     'occupe':       [0, 1, 1, 0, 0, 1, 0, 0, 0, 1] * 2,
                     'hors_service': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0] * 2, 
                     'libre':        [1, 0, 0, 1, 1, 0, 1, 1, 1, 0] * 2,
                     'nb_pdc':       [1, 1, 1, 1, 1, 1, 1, 1, 1, 1] * 2})
hyst = 3
pleine_occupation = test['occupe'].copy()
for i in range(1, hyst):
    pleine_occupation += [0] * i + list(test['occupe'])[0:len(test) - i]
f_id_pdc_itinerance = pd.Series(['aucun'] * hyst + list(test['id_pdc_itinerance'])[0:len(test) - hyst])
test['valid_po'] = f_id_pdc_itinerance == test['id_pdc_itinerance']
test['po'] = pleine_occupation > 0
test

In [ ]:
hysteresis = 3
for i in range(1, hysteresis):
    print(i)

In [ ]:
value = pd.DataFrame({'pourcent': [ 50, 80, 90, 100, 85, 75, 50, 90, 70, 100, 90, 70, 90, 50]})
value['pu_ext'] = pd.cut(value['pourcent'], [0, 80, 99, 100], labels=[1, 1.5, 2])
value

In [ ]:
import numpy as np

def hyst(x, th_lo, th_hi, initial = False):
    hi = x >= th_hi
    lo_or_hi = (x < th_lo) | hi
    ind = np.nonzero(lo_or_hi)[0]
    if not ind.size: # prevent index error if ind is empty
        return np.zeros_like(x, dtype=bool) | initial
    cnt = np.cumsum(lo_or_hi) # from 0 to len(x)
    return np.where(cnt, hi[ind[cnt-1]], initial)

In [ ]:
tmp = pd.cut(value['pourcent'], [0, 80, 99, 100], labels=[0, 0.5, 1])
pleine_occ = tmp.where(tmp != 0.5, np.where(hyst(tmp.values, 0.5, 1), 1, 0)).astype('bool')
pleine_occ